# 실제 영상 연기 탐지 v3 — Cooldown + Hysteresis + ThinSmoke 지속성 개선

`assets/capture1.avi` 영상에 YOLOv8n-cls + ThinSmokeDetectorV2 + **3단계 연속성 개선**을 적용한다.

| 항목 | 내용 |
|---|---|
| 입력 영상 | `assets/capture1.avi` (640×480, 30fps, 약 4분 30초) |
| 모델 | `runs/smoke_detector/yolov8n_cls/weights/best.pt` |
| 출력 | 프레임별 오버레이 영상 + frame_log.csv + 타임라인 |

### v2 → v3 개선 요약 (thin_smoke_continuity_fix.md 반영)

| 문제 (v2) | 해결 (v3) | 단계 |
|---|---|---|
| 연기 구간에서 탐지 on/off 반복 | **Cooldown**: smoke 판정 후 최소 1.5초(45프레임) 강제 유지 | 1단계 |
| ThinSmoke persist 조건이 너무 까다로움 | **PERSIST_WINDOW 30f, 50% 조건**으로 완화 | 2단계 |
| 진입/해제 조건이 동일 → 경계선 flickering | **Hysteresis**: 해제는 30프레임 연속 3조건 충족 시만 허용 | 3단계 |

> **원칙: 오탐(FP) 허용, 미탐(FN) 불허**

## 0. 경로 및 파라미터 설정

In [ ]:
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from collections import deque
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────────────
BASE_DIR   = Path("..").resolve()
VIDEO_PATH = BASE_DIR / "assets" / "capture1.avi"
MODEL_PATH = BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "weights" / "best.pt"
OUT_DIR    = BASE_DIR / "runs" / "smoke_detector" / "video_result_v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_VIDEO  = OUT_DIR / "capture1_result_v3.avi"
LOG_CSV    = OUT_DIR / "frame_log_v3.csv"

# ── 1단계: YOLO 탐지 파라미터 ────────────────────────────
SMOKE_THR   = 0.40  # raw confidence 임계값
VOTE_WINDOW = 5     # sliding vote 창 크기
VOTE_K      = 3     # 이 중 몇 개 이상이면 smoke 판정

# ── 2단계: ThinSmoke v2 파라미터 ─────────────────────────
WARMUP_FRAMES       = 300  # 초기 베이스라인 워밍업 (10초)
BASELINE_WINDOW     = 300  # rolling percentile 창 크기
BASELINE_PCT        = 80   # 베이스라인 percentile (80th = 선명한 정상)
SHARP_SMOOTH_WINDOW = 30   # 선명도 rolling median 창 (1초)
SHARP_DROP_RATIO    = 0.50 # smooth_sharpness < 베이스라인 × 0.50
SAT_DROP_RATIO      = 0.82 # saturation < 베이스라인 × 0.82
YOLO_UNCERTAIN_LO   = 0.00 # 하한 없음 (실제 보호 = 이중 게이트)
YOLO_UNCERTAIN_HI   = 0.40 # YOLO가 확신(>0.40)이면 보조 탐지 불필요

# ── [v3] ThinSmoke 지속성 조건 완화 (2단계) ──────────────
# v2: PERSIST_WINDOW=15, PERSIST_THRESH=12 (80%)
# v3: PERSIST_WINDOW=30, PERSIST_THRESH=15 (50%) — flickering 구간 대응
PERSIST_WINDOW = 30  # 지속성 확인 창 (v2: 15 → v3: 30)
PERSIST_THRESH = 15  # 30프레임 중 15개 이상 (50%) — 조건 완화

# ── [v3] Cooldown 파라미터 (1단계) ───────────────────────
# smoke 판정이 한 번 발생하면 최소 COOLDOWN_FRAMES 동안 강제 유지
COOLDOWN_FRAMES = 45  # 30fps 기준 약 1.5초

# ── [v3 수정] Hysteresis 파라미터 (3단계) ────────────────
# 진입: WARMUP_FRAMES 이후에만 허용 (워밍업 중 오탐 잠금 방지)
# 해제: EXIT_CONSECUTIVE 프레임 연속 raw_smoke=False이면 해제
#
# [버그 수정] 구버전 3조건(sharp_ratio/sat_ratio) 제거 이유:
#   1. 워밍업 중 sharp_ratio=None → 해제 조건 항상 실패 → smoke_state 영구 잠금
#   2. 전체 영상에서 3조건 동시 충족: 102건 뿐 (연속 30회: 0건)
#   → raw_smoke 신호만으로 해제 판단 (단순하고 실제로 동작함)
EXIT_CONSECUTIVE = 15   # 15프레임 연속 raw_smoke=False이면 해제 (약 0.5초)

# ── 영상 정보 및 모델 로드 ────────────────────────────────
cap          = cv2.VideoCapture(str(VIDEO_PATH))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS          = cap.get(cv2.CAP_PROP_FPS)
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"입력 영상     : {VIDEO_PATH.name}")
print(f"해상도        : {W} x {H}  /  FPS: {FPS:.2f}  /  총 {TOTAL_FRAMES}프레임 ({TOTAL_FRAMES/FPS:.1f}초)")
print(f"YOLO Threshold: {SMOKE_THR}  /  Vote: {VOTE_WINDOW}중 {VOTE_K}")
print(f"ThinSmoke v2  : warmup={WARMUP_FRAMES}f  window={BASELINE_WINDOW}f  pct={BASELINE_PCT}th")
print(f"              : sharp_drop<{SHARP_DROP_RATIO}  sat_drop<{SAT_DROP_RATIO}")
print(f"              : persist {PERSIST_THRESH}/{PERSIST_WINDOW}프레임  [v3 완화: 80%->50%]")
print(f"[v3] Cooldown : {COOLDOWN_FRAMES}프레임 강제 유지 (약 {COOLDOWN_FRAMES/FPS:.1f}초)")
print(f"[v3 수정] Hysteresis: 해제={EXIT_CONSECUTIVE}f 연속 raw_smoke=False  [ratio 조건 제거]")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"best.pt 없음: {MODEL_PATH}\ntrain_yolo.ipynb 먼저 실행하세요.")

model     = YOLO(str(MODEL_PATH))
smoke_idx = [k for k, v in model.names.items() if v == "smoke"][0]
print(f"모델 로드     : {MODEL_PATH}")


## 0-1. ThinSmokeDetectorV2 클래스 정의

YOLO가 확신하지 못하는 구간(`conf < 0.40`)에서 영상 품질 지표로 얇은 연기를 보완 탐지한다.

```
탐지 조건 (모두 동시 만족):
  1. YOLO conf < 0.40 (YOLO가 아직 확신 없음)
  2. smooth_sharpness < baseline_80th × 0.50  (선명도 50% 이상 하락)
  3. saturation < baseline_80th × 0.82        (채도 18% 이상 하락)
  4. 위 두 조건이 30프레임 중 15프레임 이상 지속  ← [v3 완화]
```

In [ ]:
def extract_features(frame: np.ndarray) -> tuple:
    """
    Laplacian Sharpness + HSV Saturation 추출.
    - sharpness : Laplacian variance (선명도, 연기 시 감소)
    - saturation: HSV S채널 평균 (채도, 연기 시 감소)
    """
    gray      = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    hsv        = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    saturation = float(hsv[:, :, 1].mean())
    return sharpness, saturation


class RollingPercentileBaseline:
    """
    최근 N프레임의 P번째 percentile을 베이스라인으로 사용.
    EMA 대비: 초반 고변동 구간에 오염되지 않고, alpha 튜닝 불필요.
    """
    def __init__(self, window: int = 300, percentile: int = 80):
        self.buf        = deque(maxlen=window)
        self.percentile = percentile

    def update(self, value: float):
        self.buf.append(value)

    def get_baseline(self):
        """버퍼가 창 절반 이상 차면 percentile 반환, 아니면 None"""
        if len(self.buf) < (self.buf.maxlen or 1) // 2:
            return None
        return float(np.percentile(list(self.buf), self.percentile))


class ThinSmokeDetectorV2:
    """
    v1의 97% 오탐 문제를 해결한 보조 탐지기.

    v3 변경사항:
    - PERSIST_WINDOW: 15 → 30 (더 긴 구간 관찰)
    - PERSIST_THRESH: 12/15 (80%) → 15/30 (50%) (조건 완화)
    → 얇은 연기 flickering 구간에서 탐지 실패 방지
    """
    def __init__(self):
        self.sharp_baseline   = RollingPercentileBaseline(BASELINE_WINDOW, BASELINE_PCT)
        self.sat_baseline     = RollingPercentileBaseline(BASELINE_WINDOW, BASELINE_PCT)
        self.persist_buf      = deque(maxlen=PERSIST_WINDOW)  # v3: 30
        self.sharp_smooth_buf = deque(maxlen=SHARP_SMOOTH_WINDOW)
        self.frame_count      = 0

    def update(self, frame: np.ndarray, yolo_conf: float) -> tuple:
        """
        Args:
            frame    : BGR 프레임
            yolo_conf: YOLO smoke confidence (0~1)
        Returns:
            thin_smoke: True면 보조 탐지 발동
            info      : 디버그 정보 dict
        """
        self.frame_count += 1
        sharpness, saturation = extract_features(frame)

        # 항상 업데이트 (워밍업 중에도 베이스라인 축적)
        self.sharp_smooth_buf.append(sharpness)
        self.sharp_baseline.update(sharpness)
        self.sat_baseline.update(saturation)

        smooth_sharpness = float(np.median(list(self.sharp_smooth_buf)))

        info = {
            "sharpness": smooth_sharpness,
            "saturation": saturation,
            "sharp_base": None, "sat_base": None,
            "sharp_ratio": None, "sat_ratio": None,
            "both_low": False, "persist_cnt": 0,
        }

        # 워밍업 중에는 탐지 비활성
        if self.frame_count < WARMUP_FRAMES:
            self.persist_buf.append(0)
            return False, info

        # YOLO가 이미 smoke에 확신 → 보조 탐지 불필요
        if not (YOLO_UNCERTAIN_LO <= yolo_conf <= YOLO_UNCERTAIN_HI):
            self.persist_buf.append(0)
            return False, info

        sharp_base = self.sharp_baseline.get_baseline()
        sat_base   = self.sat_baseline.get_baseline()
        if sharp_base is None or sat_base is None or sharp_base < 1.0 or sat_base < 1.0:
            self.persist_buf.append(0)
            return False, info

        # 동시 하락 감지
        sharp_ratio = smooth_sharpness / sharp_base
        sat_ratio   = saturation / sat_base
        both_low    = (sharp_ratio < SHARP_DROP_RATIO) and (sat_ratio < SAT_DROP_RATIO)

        info.update({
            "sharp_base": sharp_base, "sat_base": sat_base,
            "sharp_ratio": sharp_ratio, "sat_ratio": sat_ratio,
            "both_low": both_low,
        })

        self.persist_buf.append(1 if both_low else 0)
        persist_cnt         = sum(self.persist_buf)
        info["persist_cnt"] = persist_cnt

        # v3: 30프레임 중 15개 이상 (50%)
        thin_smoke = (len(self.persist_buf) == PERSIST_WINDOW and persist_cnt >= PERSIST_THRESH)
        return thin_smoke, info


def draw_overlay_v3(frame, final_label, yolo_raw, yolo_conf, vote_cnt,
                    thin_smoke, thin_info, smoke_state, cooldown_counter,
                    frame_idx, total, elapsed):
    """상단 바에 최종 판정 + YOLO raw + ThinSmoke + Cooldown/Hysteresis 상태 표시."""
    h, w      = frame.shape[:2]
    is_smoke  = (final_label == "smoke")
    is_thin   = thin_smoke and not (yolo_conf >= SMOKE_THR)
    is_cooldown = (not smoke_state) and (cooldown_counter > 0)  # cooldown 강제 유지 중
    bar_color = (0, 0, 200) if is_smoke else (0, 180, 0)

    cv2.rectangle(frame, (0, 0), (w, 90), bar_color, -1)

    # 상태 태그 구성
    tag = "[SMOKE]"
    if not is_smoke:
        tag = "[NO SMOKE]"
    elif is_cooldown:
        tag += " (cooldown)"
    elif is_thin:
        tag += " (thin)"

    cv2.putText(frame, f"{tag}  vote={vote_cnt}/{VOTE_WINDOW}",
                (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"YOLO: {yolo_conf:.2f} ({yolo_raw})",
                (10, 46), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (220, 220, 220), 1, cv2.LINE_AA)

    if thin_info["sharp_ratio"] is not None:
        thin_txt = (f"ThinSmoke: sharp_r={thin_info['sharp_ratio']:.2f} "
                    f"sat_r={thin_info['sat_ratio']:.2f} "
                    f"persist={thin_info['persist_cnt']}/{PERSIST_WINDOW}")
    else:
        thin_txt = f"ThinSmoke: warmup... ({max(0, WARMUP_FRAMES - frame_idx)}f left)"
    cv2.putText(frame, thin_txt, (10, 68),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (200, 230, 255), 1, cv2.LINE_AA)

    # cooldown 카운터 표시
    cd_txt = f"Cooldown: {cooldown_counter:3d}f  |  Hysteresis: {'SMOKE' if smoke_state else 'CLEAR'}"
    cv2.putText(frame, cd_txt, (10, 85),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 180), 1, cv2.LINE_AA)

    cv2.putText(frame, f"Frame {frame_idx}/{total}  {elapsed:.1f}s",
                (w - 270, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return frame


print("클래스 정의 완료: extract_features / RollingPercentileBaseline / ThinSmokeDetectorV2")
print(f"[v3] PERSIST_WINDOW={PERSIST_WINDOW}, PERSIST_THRESH={PERSIST_THRESH} (50%)")

## 1. 영상 전체 추론 → 결과 영상 저장

**v3 추론 파이프라인** (순서 중요):

```
① YOLO 추론
② Sliding window vote (YOLO)
③ ThinSmoke v2 보조 탐지 (PERSIST 완화 적용)
④ Hysteresis 판정 (진입 쉽게, 해제 어렵게)
   └ smoke_state: 30프레임 연속 3조건 충족 시만 해제
⑤ Cooldown (smoke_state 기준으로 최소 45프레임 강제 유지)
⑥ 오버레이 저장 + CSV 로그
```

> **순서 주의**: Cooldown은 반드시 Hysteresis 결과(`smoke_state`) 기준으로 동작해야 이중 억제가 발생하지 않음.

In [ ]:
cap_read = cv2.VideoCapture(str(VIDEO_PATH))
writer   = cv2.VideoWriter(str(OUT_VIDEO), cv2.VideoWriter_fourcc(*"XVID"), FPS, (W, H))
vote_buf = deque(maxlen=VOTE_WINDOW)
detector = ThinSmokeDetectorV2()

# ── [v3] 상태 변수 초기화 ─────────────────────────────────
cooldown_counter       = 0
cooldown_forced_count  = 0   # cooldown으로 강제 유지된 프레임 수
smoke_state            = False  # 현재 확정 smoke 상태 (Hysteresis)
clear_streak           = 0      # 연속 raw_smoke=False 카운터
hysteresis_held_count  = 0      # smoke_state=True인데 raw_smoke=False인 프레임

yolo_smoke_cnt  = 0
thin_smoke_cnt  = 0
thin_raw_cnt    = 0
final_smoke_cnt = 0
frame_idx       = 0
ms_list         = []
start_wall      = time.perf_counter()

with open(LOG_CSV, "w", encoding="utf-8") as f:
    f.write("frame,sec,yolo_conf,yolo_raw,thin_smoke,smoke_state,cooldown,final,"
            "sharp,sat,sharp_ratio,sat_ratio,persist\n")

print("추론 시작 (v3 수정)...")
while True:
    ret, frame = cap_read.read()
    if not ret:
        break
    frame_idx += 1

    # ① YOLO 추론
    t0        = time.perf_counter()
    results   = model.predict(frame, verbose=False, imgsz=224)
    ms_list.append((time.perf_counter() - t0) * 1000)

    yolo_conf = float(results[0].probs.data[smoke_idx])
    yolo_raw  = "smoke" if yolo_conf >= SMOKE_THR else "no_smoke"

    # ② Sliding window vote
    vote_buf.append(1 if yolo_raw == "smoke" else 0)
    vote_cnt  = sum(vote_buf)
    yolo_vote = vote_cnt >= VOTE_K
    if yolo_vote:
        yolo_smoke_cnt += 1

    # ③ ThinSmoke v2 보조 탐지 (PERSIST_WINDOW=30, PERSIST_THRESH=15 적용)
    thin_smoke, thin_info = detector.update(frame, yolo_conf)
    if thin_smoke:
        thin_raw_cnt += 1

    # ④ [v3 수정] Hysteresis 판정 ────────────────────────────
    raw_smoke = yolo_vote or thin_smoke

    if not smoke_state:
        # 진입 조건: raw_smoke=True 이고 워밍업 완료 이후에만 진입
        # [버그 수정] 워밍업 중(frame_idx < WARMUP_FRAMES)에는 진입하지 않음:
        #   - 워밍업 중에는 sharp_ratio=None → 해제 조건 평가 불가
        #   - 진입 후 해제가 안 되면 smoke_state가 영상 전체에서 True로 잠김
        if raw_smoke and frame_idx >= WARMUP_FRAMES:
            smoke_state  = True
            clear_streak = 0
    else:
        # 해제 조건: raw_smoke=False가 EXIT_CONSECUTIVE 프레임 연속
        # [버그 수정] 구버전의 3조건(sharp/sat ratio) 제거:
        #   - 전체 영상에서 3조건 동시 충족: 102건뿐, 연속 30회: 0건 → 해제 불가
        #   - raw_smoke 신호 기반 해제로 단순화
        if not raw_smoke:
            clear_streak += 1
            if clear_streak >= EXIT_CONSECUTIVE:
                smoke_state  = False
                clear_streak = 0
        else:
            # raw_smoke 재탐지 → streak 리셋
            if clear_streak > 0:
                hysteresis_held_count += clear_streak
            clear_streak = 0

    # ⑤ [v3] Cooldown: smoke_state 기준 최소 COOLDOWN_FRAMES 강제 유지
    # smoke_state=True 이면 cooldown을 항상 COOLDOWN_FRAMES으로 갱신
    # smoke_state=False이면 카운트다운 → 0이 되어야 최종 no_smoke
    final_smoke = smoke_state
    if final_smoke:
        cooldown_counter = COOLDOWN_FRAMES
    elif cooldown_counter > 0:
        final_smoke      = True
        cooldown_counter -= 1
        cooldown_forced_count += 1

    # 통계 집계
    if final_smoke:
        final_smoke_cnt += 1
    if thin_smoke and not yolo_vote:
        thin_smoke_cnt += 1
    final_label = "smoke" if final_smoke else "no_smoke"

    # ⑥ 오버레이 저장
    elapsed   = time.perf_counter() - start_wall
    out_frame = draw_overlay_v3(
        frame.copy(), final_label, yolo_raw, yolo_conf, vote_cnt,
        thin_smoke, thin_info, smoke_state, cooldown_counter,
        frame_idx, TOTAL_FRAMES, elapsed
    )
    writer.write(out_frame)

    # CSV 로그
    with open(LOG_CSV, "a", encoding="utf-8") as f:
        sec = frame_idx / FPS
        sr  = f"{thin_info['sharp_ratio']:.3f}" if thin_info["sharp_ratio"] is not None else ""
        sar = f"{thin_info['sat_ratio']:.3f}"   if thin_info["sat_ratio"]   is not None else ""
        f.write(
            f"{frame_idx},{sec:.2f},{yolo_conf:.4f},{yolo_raw},"
            f"{int(thin_smoke)},{int(smoke_state)},{cooldown_counter},{final_label},"
            f"{thin_info['sharpness']:.1f},{thin_info['saturation']:.2f},"
            f"{sr},{sar},{thin_info['persist_cnt']}\n"
        )

    if frame_idx % 500 == 0 or frame_idx == TOTAL_FRAMES:
        pct    = frame_idx / TOTAL_FRAMES * 100
        avg_ms = np.mean(ms_list[-500:])
        print(
            f"  [{pct:5.1f}%] {frame_idx}/{TOTAL_FRAMES}  "
            f"avg {avg_ms:.1f}ms  "
            f"YOLO_smoke={yolo_smoke_cnt}({yolo_smoke_cnt/frame_idx*100:.1f}%)  "
            f"ThinOnly={thin_smoke_cnt}  "
            f"Final={final_smoke_cnt}({final_smoke_cnt/frame_idx*100:.1f}%)"
        )

cap_read.release()
writer.release()

total_sec = time.perf_counter() - start_wall
total     = max(frame_idx, 1)

print("\n" + "=" * 65)
print(f"  처리 프레임     : {frame_idx}장  ({total_sec:.1f}초)")
print(f"  평균 속도       : {np.mean(ms_list):.1f} ms/frame  ({1000/np.mean(ms_list):.1f} FPS)")
print(f"  YOLO smoke      : {yolo_smoke_cnt}장 ({yolo_smoke_cnt/total*100:.1f}%)")
print(f"  ThinSmoke 발화  : {thin_raw_cnt}장 ({thin_raw_cnt/total*100:.1f}%)")
print(f"  ThinSmoke 추가  : {thin_smoke_cnt}장 ({thin_smoke_cnt/total*100:.1f}%)  <- 보조 탐지만")
print(f"  최종 smoke      : {final_smoke_cnt}장 ({final_smoke_cnt/total*100:.1f}%)")
print(f"  최종 no_smoke   : {total-final_smoke_cnt}장 ({(total-final_smoke_cnt)/total*100:.1f}%)")
print()
print(f"  [v3] Cooldown 강제 유지   : {cooldown_forced_count}프레임")
print(f"  [v3] Hysteresis 해제 유예 : {hysteresis_held_count}프레임")
print(f"  저장 영상       : {OUT_VIDEO}")
print(f"  프레임 로그     : {LOG_CSV}")
print("=" * 65)


## 2. 시간대별 smoke 탐지 타임라인

YOLO 기여분(파란색), ThinSmoke 보조 탐지(주황색), Cooldown/Hysteresis 강제 유지(초록색)를 구분하여 시각화한다.

In [ ]:
df      = pd.read_csv(LOG_CSV)
bin_sec = 5

df["bin"] = (df["sec"] // bin_sec).astype(int)
timeline  = df.groupby("bin").agg(
    yolo_ratio   = ("yolo_raw",    lambda x: (x == "smoke").mean()),
    final_ratio  = ("final",       lambda x: (x == "smoke").mean()),
    thin_ratio   = ("thin_smoke",  "mean"),
    state_ratio  = ("smoke_state", "mean"),  # hysteresis 확정 smoke
    cd_ratio     = ("cooldown",    lambda x: (x > 0).mean()),  # cooldown 활성 비율
).reset_index()

n_bins   = len(timeline)
x_labels = [f"{int(b*bin_sec)//60}:{int(b*bin_sec)%60:02d}" for b in timeline["bin"]]

fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

# 상단: YOLO + ThinSmoke 기여 분리
axes[0].bar(range(n_bins), timeline["yolo_ratio"],  color="steelblue", width=0.85, label="YOLO vote")
axes[0].bar(range(n_bins), timeline["thin_ratio"],  bottom=timeline["yolo_ratio"],
            color="orange", width=0.85, alpha=0.8,  label="ThinSmoke 추가분")
axes[0].axhline(0.5, color="gray", ls="--", lw=1)
axes[0].set_ylabel("Smoke Ratio")
axes[0].set_title("v3 Detection — YOLO + ThinSmoke (5s bins)")
axes[0].set_ylim(0, 1.1)
axes[0].legend(loc="upper right")

# 중단: Hysteresis smoke_state
axes[1].bar(range(n_bins), timeline["state_ratio"], color="mediumpurple", width=0.85, label="Hysteresis smoke_state")
axes[1].axhline(0.5, color="gray", ls="--", lw=1)
axes[1].set_ylabel("State Ratio")
axes[1].set_title("Hysteresis 확정 smoke_state (보라색)")
axes[1].set_ylim(0, 1.1)
axes[1].legend(loc="upper right")

# 하단: 최종 판정 (Cooldown 포함)
colors = ["tomato" if r >= 0.5 else "mediumseagreen" for r in timeline["final_ratio"]]
axes[2].bar(range(n_bins), timeline["final_ratio"], color=colors, width=0.85)
axes[2].axhline(0.5, color="gray", ls="--", lw=1)
axes[2].set_ylabel("Final Smoke Ratio")
axes[2].set_title("Final Decision Timeline — Cooldown+Hysteresis 적용 (red=smoke, green=no_smoke)")
axes[2].set_ylim(0, 1.1)
axes[2].set_xticks(range(0, n_bins, max(1, n_bins // 20)))
axes[2].set_xticklabels(
    [x_labels[i] for i in range(0, n_bins, max(1, n_bins // 20))],
    rotation=45, ha="right"
)
axes[2].set_xlabel("Time (min:sec)")

plt.tight_layout()
timeline_path = OUT_DIR / "smoke_timeline_v3.png"
plt.savefig(str(timeline_path), dpi=150)
plt.show()
print(f"타임라인 저장: {timeline_path}")

## 3. ThinSmoke 탐지 구간 상세 분석

ThinSmoke가 발화한 프레임과 그 전후 문맥을 확인한다.

In [ ]:
thin_only = df[(df["thin_smoke"] == 1) & (df["yolo_raw"] == "no_smoke")]

print(f"ThinSmoke 보조 탐지 프레임 수: {len(thin_only)}장")
print(f"FP 검증 (0~30초 구간 오탐): {len(df[(df['sec'] <= 30) & (df['thin_smoke'] == 1)])}건  ← 0이어야 함")
print()

if len(thin_only) > 0:
    print("[ThinSmoke 발화 구간]")
    print(thin_only[["sec", "yolo_conf", "sharp_ratio", "sat_ratio", "persist"]].to_string(index=False))

## 4. 연속성 개선 효과 검증

thin_smoke_continuity_fix.md 검증 항목을 확인한다.

In [ ]:
# ── 검증 항목 1: 0~30초 구간 오탐 수 ─────────────────────
fp_count = len(df[(df["sec"] <= 30) & (df["final"] == "smoke")])

# ── 검증 항목 2: 얇은 연기 구간(31초~) 연속성 분석 ───────
smoke_after_31 = df[df["sec"] > 31].copy()
smoke_after_31["is_smoke"] = (smoke_after_31["final"] == "smoke").astype(int)

# 연속 탐지 구간(run) 분석
def find_runs(series):
    """True/False 시퀀스에서 연속 구간 길이 목록 반환"""
    runs = []
    if len(series) == 0:
        return runs
    current_val = series.iloc[0]
    current_len = 1
    for v in series.iloc[1:]:
        if v == current_val:
            current_len += 1
        else:
            runs.append((current_val, current_len))
            current_val = v
            current_len = 1
    runs.append((current_val, current_len))
    return runs

runs    = find_runs(smoke_after_31["is_smoke"])
# smoke 구간 중 끊김(no_smoke) 길이
gaps    = [length for val, length in runs if val == 0]
smokes  = [length for val, length in runs if val == 1]

# ── 검증 항목 3: ThinSmoke 발화 프레임 수 ────────────────
thin_count_total = len(df[df["thin_smoke"] == 1])

# ── 검증 항목 4: 최종 smoke 비율 ─────────────────────────
final_smoke_ratio = (df["final"] == "smoke").mean() * 100

print("=" * 55)
print("  [v3 검증 결과] thin_smoke_continuity_fix.md 기준")
print("=" * 55)
print(f"  ① 0~30초 오탐 수       : {fp_count}건  (목표: 0건)  {'✓ PASS' if fp_count == 0 else '✗ FAIL'}")
print(f"  ② smoke 구간 평균 연속 : {np.mean(smokes):.1f}프레임  (길수록 연속성 ↑)")
print(f"     smoke 구간 내 끊김  : 평균 {np.mean(gaps):.1f}f / 최대 {max(gaps) if gaps else 0}f  (짧을수록 연속성 ↑)")
print(f"  ③ ThinSmoke 발화 수    : {thin_count_total}프레임  (v2: 30장, 증가 예상)")
print(f"  ④ 최종 smoke 비율      : {final_smoke_ratio:.1f}%  (v2: 44.5%, 소폭 증가 허용)")
print()
print(f"  [v3 신규] Cooldown 강제  : {cooldown_forced_count}프레임")
print(f"  [v3 신규] Hysteresis 유지: {hysteresis_held_count}프레임")
print()

# ── smoke 구간 탐지 끊김 시각화 ──────────────────────────
fig, ax = plt.subplots(figsize=(18, 3))
ax.fill_between(smoke_after_31["sec"], smoke_after_31["is_smoke"],
                alpha=0.5, color="tomato", label="final smoke")
ax.set_xlabel("Time (sec)")
ax.set_ylabel("Smoke (1=Yes)")
ax.set_title("v3 연기 구간(31초~) 연속성 — 끊김이 없을수록 개선")
ax.set_ylim(-0.1, 1.2)
ax.legend()
continuity_path = OUT_DIR / "smoke_continuity_v3.png"
plt.tight_layout()
plt.savefig(str(continuity_path), dpi=150)
plt.show()
print(f"연속성 그래프 저장: {continuity_path}")